# 03 — File Pipeline
Explore → Extract → Analyse → Ingest

This notebook covers the **file** object type from the raw Backyard dump.

**Confirmed graph model:**
- `:File` node — non-image files only (`file_is_image == False`)
- **Properties:** `id`, `file_name`, `file_type`, `file_mime_type`, `file_provider`, `is_active`, `is_deleted`, `file_url`, `_aggregated_locations`, `createddate`, `lastmodifieddate`
- **Relationships:**
  - `(People)-[:CREATED]->(File)` via `createdbyid`
  - `(File)-[:BELONGS_TO_SITE]->(Site)` via `file_site_mapping` array (one edge per site ID)
- Orphan files (empty `file_site_mapping`) are ingested without site edges
- Missing site IDs are logged and silently skipped during ingestion
- `file_content_mappings` is stored in normalized `files.jsonl` as a clean list of content IDs; it is deferred to the content pipeline and not ingested in Neo4j here

## 0 · Setup

In [1]:
import json, sys
from pathlib import Path
from collections import Counter

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import RAW_FILE, NORMALIZED_DIR, NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD

print(f"Project root : {PROJECT_ROOT}")
print(f"Raw file     : {RAW_FILE}")
print(f"Raw file exists: {RAW_FILE.exists()}")

Project root : /home/ubuntu/graph_experiments
Raw file     : /home/ubuntu/graph_experiments/data/raw/backyard_dump_19022026_172648.jsonl
Raw file exists: True


## 1 · Explore Raw Data

In [2]:
# Load all file records from the raw dump, split images from non-images
all_file_items = []
with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        if obj.get("object_type") == "file":
            all_file_items.append(obj)

image_items    = [f for f in all_file_items if f.get("file_is_image") is True]
file_items     = [f for f in all_file_items if f.get("file_is_image") is not True]

print(f"Total file records : {len(all_file_items)}")
print(f"  → images  (excluded from analysis) : {len(image_items)}")
print(f"  → non-image (used for all analysis) : {len(file_items)}")

Total file records : 7877
  → images  (excluded from analysis) : 7322
  → non-image (used for all analysis) : 555


In [3]:
# Save first 10 non-image samples to data/debug/sample_files.jsonl
sample_path = PROJECT_ROOT / "data" / "debug" / "sample_files.jsonl"
sample_path.parent.mkdir(parents=True, exist_ok=True)
with open(sample_path, "w") as out:
    for item in file_items[:10]:
        out.write(json.dumps(item) + "\n")
print(f"Saved {min(10, len(file_items))} non-image samples → {sample_path}")

# Pretty-print first non-image record (truncate large text fields)
sample = dict(file_items[0])
for big_field in ("file_data", "file_transcript", "extracted_info"):
    if sample.get(big_field) and len(str(sample[big_field])) > 300:
        sample[big_field] = str(sample[big_field])[:300] + " ...[truncated]"
print("\n--- Sample non-image record ---")
print(json.dumps(sample, indent=2))

Saved 10 non-image samples → /home/ubuntu/graph_experiments/data/debug/sample_files.jsonl

--- Sample non-image record ---
{
  "id": "1c148301-6d28-46b4-b833-1bda96f2bb8c",
  "is_active": true,
  "is_deleted": false,
  "file_site_mapping": [
    "39b3d853-fcc5-43cd-a508-d77f56bb0b02"
  ],
  "createddate": "2024-08-26T19:40:36.312Z",
  "createdbyid": "3a331b59-add7-4198-a9a8-78c11af00261",
  "file_provider": "intranet",
  "object_type": "file",
  "lastmodifieddate": "2024-08-26T19:40:36.312Z",
  "file_s3_path": "191e03de-fb5e-4b01-b9fd-faf1873292a6/content/restricted/1c148301-6d28-46b4-b833-1bda96f2bb8c",
  "file_name": "Ragan -LumApps-Webinar-8.22.24.pdf",
  "file_size": 4099081,
  "file_mime_type": "application/pdf",
  "file_url": "/content/r/1c148301-6d28-46b4-b833-1bda96f2bb8c",
  "file_site_id": "39b3d853-fcc5-43cd-a508-d77f56bb0b02",
  "file_site_name": "Strategy & Value Consulting",
  "file_site_type": "public",
  "file_site_is_active": true,
  "file_content_id": "28073",
  "file

In [4]:
# Key presence across non-image file records
all_keys = set()
for item in file_items:
    all_keys.update(item.keys())

n = len(file_items)
print(f"Analysing {n} non-image file records\n")
print(f"{'Key':<45} {'Present':>8} {'Missing':>8}")
print("-" * 67)
for k in sorted(all_keys):
    present = sum(1 for s in file_items if k in s)
    missing = n - present
    flag = "  ⚠️" if missing > 0 else ""
    print(f"{k:<45} {present:>5}/{n:<5} {missing:>5}{flag}")

Analysing 555 non-image file records

Key                                            Present  Missing
-------------------------------------------------------------------
_aggregated_locations                           341/555     214  ⚠️
_s3_orchestrator_processing_bucket_key          550/555       5  ⚠️
_s3_orchestrator_processing_file_key            550/555       5  ⚠️
autocomplete                                    555/555       0
createdbyid                                     555/555       0
createddate                                     555/555       0
extracted_info                                    4/555     551  ⚠️
file_author                                     555/555       0
file_content_id                                 302/555     253  ⚠️
file_content_mapping                             45/555     510  ⚠️
file_content_type                               555/555       0
file_creation_date                              555/555       0
file_data                             

### 1.1 Null values
Keys that are **present** but have `None` as value (different from missing keys above).

In [5]:
# Check for None values (key present but value is None)
n = len(file_items)
print(f"{'Key':<45} {'Null':>6} / {n}")
print("-" * 57)
for k in sorted(all_keys):
    nulls = sum(1 for s in file_items if k in s and s[k] is None)
    if nulls > 0:
        print(f"{k:<45} {nulls:>6}")
print("\n(Only keys with at least 1 null shown)")

Key                                             Null / 555
---------------------------------------------------------
_s3_orchestrator_processing_bucket_key           210
_s3_orchestrator_processing_file_key             210
file_author                                      555
file_content_id                                    6
file_content_type                                212
file_creation_date                               555
file_data                                        212
file_external_id                                 357
file_language                                    555
file_mime_type                                     3
file_page_count                                  238
file_s3_path                                     200
file_size                                          2
file_thumbnail_url                               434
file_title                                       555
file_url                                         200

(Only keys with at least 1 null sh

### 1.2 Active / Deleted distribution

In [6]:
print("is_active:",  Counter(s.get("is_active")  for s in file_items))
print("is_deleted:", Counter(s.get("is_deleted") for s in file_items))

is_active: Counter({True: 530, False: 25})
is_deleted: Counter({False: 473, True: 82})


### 1.3 file_type, file_mime_type, file_provider

In [7]:
# file_type distribution
print("=== file_type ===")
for k, v in Counter(s.get("file_type") for s in file_items).most_common():
    print(f"  {repr(k)}: {v}")

print()
print("=== file_mime_type (top 20) ===")
for k, v in Counter(s.get("file_mime_type") for s in file_items).most_common(20):
    print(f"  {repr(k)}: {v}")

print()
print("=== file_provider ===")
print(Counter(s.get("file_provider") for s in file_items))

=== file_type ===
  'PDF': 273
  '': 204
  'PPTX': 29
  'DOCX': 11
  'XLSX': 8
  'TXT': 5
  'PPT': 4
  'ZIP': 4
  'HTML': 3
  'XLS': 3
  'PHP': 2
  'WAV': 2
  'JS': 2
  'CSV': 1
  'POTX': 1
  'JSON': 1
  'MPGA': 1
  'GSLIDES': 1

=== file_mime_type (top 20) ===
  'application/pdf': 273
  'PPTX': 130
  'application/vnd.openxmlformats-officedocument.presentationml.presentation': 29
  'GDOC': 21
  'GSLIDES': 19
  'GSHEET': 12
  'application/vnd.openxmlformats-officedocument.wordprocessingml.document': 11
  'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet': 8
  None: 7
  'text/plain': 5
  'MP4': 5
  'application/vnd.ms-powerpoint': 4
  'application/zip': 4
  'text/html': 3
  'DOCX': 3
  'QT': 3
  'application/vnd.ms-excel': 3
  'application/x-httpd-php': 2
  'PDF': 2
  'audio/wave': 2

=== file_provider ===
Counter({'intranet': 357, 'googledrive': 198})


### 1.4 Site association — `file_site_id` / `file_site_name`
Check the site name→id mapping for conflicts and distribution.

In [8]:
# file_site_mapping cardinality
no_site   = sum(1 for s in file_items if not s.get("file_site_mapping"))
one_site  = sum(1 for s in file_items if len(s.get("file_site_mapping") or []) == 1)
multi_site = sum(1 for s in file_items if len(s.get("file_site_mapping") or []) > 1)
print(f"No site mapping  : {no_site}  (orphan files — ingested without BELONGS_TO_SITE edge)")
print(f"Single-site      : {one_site}")
print(f"Multi-site       : {multi_site}  (one BELONGS_TO_SITE edge per entry)")
print()

# Top sites by non-image file count
print("=== Top 15 sites by non-image file count ===")
site_counter = Counter(s.get("file_site_name") for s in file_items if s.get("file_site_name"))
for name, cnt in site_counter.most_common(15):
    print(f"  {name}: {cnt}")

No site mapping  : 6  (orphan files — ingested without BELONGS_TO_SITE edge)
Single-site      : 524
Multi-site       : 25  (one BELONGS_TO_SITE edge per entry)

=== Top 15 sites by non-image file count ===
  Strategy & Value Consulting: 130
  IT & Security Team: 57
  Marketing Team: 56
  Product Team: 54
  Professional Services: 33
  Life at Simpplr: 28
  Engineering Team: 24
  Revenue Team: 24
  Simpplr One Upgrade: 22
  Customer Experience Team: 18
  USA : 16
  India : 15
  People Managers: 14
  Finance: 13
  Solutions Consulting: 11


### 1.5 Content mapping normalization (`file_content_mappings`)
Build one unified list key from raw `file_content_id` + `file_content_mapping`, then validate null/shape/mismatch behavior.

In [9]:
def clean_string(value):
    if value is None:
        return None
    value = str(value).strip()
    if value == "" or value.startswith("@@"):
        return None
    return value


def clean_content_mapping_ids(values):
    if not isinstance(values, list):
        return []
    cleaned = []
    for value in values:
        if not isinstance(value, dict):
            continue
        content_id = clean_string(value.get("id"))
        if content_id:
            cleaned.append(content_id)
    return cleaned


def build_content_mappings(file_content_id, file_content_mapping):
    combined = []
    scalar_id = clean_string(file_content_id) if isinstance(file_content_id, str) else None
    if scalar_id:
        combined.append(scalar_id)
    combined.extend(clean_content_mapping_ids(file_content_mapping))

    seen = set()
    unique = []
    for value in combined:
        if value not in seen:
            seen.add(value)
            unique.append(value)
    return unique

content_id_nonempty = 0
raw_mapping_present_key = 0
raw_mapping_with_valid_id = 0
normalized_mappings_nonempty = 0
normalized_mappings_empty = 0
non_dict_mapping_items = 0
dict_without_valid_id = 0
mapping_lengths = Counter()

for record in file_items:
    raw_content_id = record.get("file_content_id")
    raw_mapping = record.get("file_content_mapping")

    if clean_string(raw_content_id):
        content_id_nonempty += 1
    if raw_mapping is not None:
        raw_mapping_present_key += 1

    if isinstance(raw_mapping, list):
        non_dict_mapping_items += sum(
            1 for value in raw_mapping if value is not None and not isinstance(value, dict)
        )
        dict_without_valid_id += sum(
            1
            for value in raw_mapping
            if isinstance(value, dict) and not clean_string(value.get("id"))
        )

    extracted_ids = clean_content_mapping_ids(raw_mapping)
    if extracted_ids:
        raw_mapping_with_valid_id += 1

    normalized = build_content_mappings(raw_content_id, raw_mapping)
    mapping_lengths[len(normalized)] += 1
    if normalized:
        normalized_mappings_nonempty += 1
    else:
        normalized_mappings_empty += 1

print(f"Files analysed (non-image): {len(file_items)}")
print(f"Raw file_content_id non-empty                 : {content_id_nonempty}")
print(f"Raw file_content_mapping key present          : {raw_mapping_present_key}")
print(f"Raw file_content_mapping with valid item.id   : {raw_mapping_with_valid_id}")
print(f"Normalized file_content_mappings non-empty    : {normalized_mappings_nonempty}")
print(f"Normalized file_content_mappings empty        : {normalized_mappings_empty}")
print(f"Non-dict items in raw mapping lists           : {non_dict_mapping_items}")
print(f"Dict items without valid id in raw mapping    : {dict_without_valid_id}")
print()
print("Normalized file_content_mappings length distribution (top 10):")
print(mapping_lengths.most_common(10))

Files analysed (non-image): 555
Raw file_content_id non-empty                 : 296
Raw file_content_mapping key present          : 45
Raw file_content_mapping with valid item.id   : 27
Normalized file_content_mappings non-empty    : 296
Normalized file_content_mappings empty        : 259
Non-dict items in raw mapping lists           : 0
Dict items without valid id in raw mapping    : 0

Normalized file_content_mappings length distribution (top 10):
[(1, 296), (0, 259)]


In [11]:
# Raw key comparison: file_content_id vs file_content_mapping[*].id
# Expected rule: when both are present, file_content_id should be a subset/member of mapping IDs.

both_present = 0
subset_ok = 0
subset_fail = 0

id_only = 0
mapping_only = 0
both_empty = 0

id_present_mapping_key_missing = 0
id_present_mapping_key_present_but_no_ids = 0
mapping_ids_present_id_missing = 0

subset_fail_samples = []
id_only_samples = []
mapping_only_samples = []

for rec in file_items:
    file_id = rec.get("id")

    raw_content_id = clean_string(rec.get("file_content_id"))
    raw_map_value = rec.get("file_content_mapping")
    mapping_ids = clean_content_mapping_ids(raw_map_value)

    has_id = bool(raw_content_id)
    has_mapping_ids = bool(mapping_ids)
    mapping_key_present = raw_map_value is not None

    if has_id and has_mapping_ids:
        both_present += 1
        if raw_content_id in mapping_ids:
            subset_ok += 1
        else:
            subset_fail += 1
            if len(subset_fail_samples) < 10:
                subset_fail_samples.append({
                    "file_id": file_id,
                    "file_content_id": raw_content_id,
                    "mapping_ids": mapping_ids[:5],
                })
    elif has_id and not has_mapping_ids:
        id_only += 1
        if not mapping_key_present:
            id_present_mapping_key_missing += 1
        else:
            id_present_mapping_key_present_but_no_ids += 1
        if len(id_only_samples) < 10:
            id_only_samples.append({
                "file_id": file_id,
                "file_content_id": raw_content_id,
                "raw_mapping_type": type(raw_map_value).__name__,
            })
    elif has_mapping_ids and not has_id:
        mapping_only += 1
        mapping_ids_present_id_missing += 1
        if len(mapping_only_samples) < 10:
            mapping_only_samples.append({
                "file_id": file_id,
                "mapping_ids": mapping_ids[:5],
            })
    else:
        both_empty += 1

print("=== Raw key comparison (non-image files) ===")
print(f"Total records: {len(file_items)}")
print(f"Both present (id + mapping_ids): {both_present}")
print(f"  subset OK   (id in mapping_ids): {subset_ok}")
print(f"  subset FAIL (id not in mapping) : {subset_fail}")
print()
print(f"id only (id present, mapping_ids empty): {id_only}")
print(f"  ├─ mapping key missing (None): {id_present_mapping_key_missing}")
print(f"  └─ mapping key present but no valid ids: {id_present_mapping_key_present_but_no_ids}")
print(f"mapping only (mapping_ids present, id empty): {mapping_only}")
print(f"both empty/missing: {both_empty}")

if subset_fail_samples:
    print("\nSubset FAIL samples:")
    for row in subset_fail_samples:
        print(row)

if id_only_samples:
    print("\nID-only samples:")
    for row in id_only_samples:
        print(row)

if mapping_only_samples:
    print("\nMapping-only samples:")
    for row in mapping_only_samples:
        print(row)# Print one raw file_content_mapping item example (object shape)
# and one problematic payload (non-dict item or dict without valid id), if any.

example_found = False
problem_found = False

for rec in file_items:
    raw_map = rec.get("file_content_mapping")
    if not isinstance(raw_map, list):
        continue

    for item in raw_map:
        if isinstance(item, dict) and not example_found:
            print("Example mapping object:")
            print(f"  file_id: {rec.get('id')}")
            print(f"  item   : {item}")
            print(f"  id     : {item.get('id')!r}")
            example_found = True

        invalid = (item is not None and not isinstance(item, dict)) or (
            isinstance(item, dict) and not clean_string(item.get("id"))
        )
        if invalid and not problem_found:
            print("\nProblematic mapping payload:")
            print(f"  file_id: {rec.get('id')}")
            print(f"  item   : {item!r}")
            print(f"  type   : {type(item).__name__}")
            problem_found = True

        if example_found and problem_found:
            break

    if example_found and problem_found:
        break

if not example_found:
    print("No mapping object examples found.")
if not problem_found:
    print("No problematic mapping payload found.")

=== Raw key comparison (non-image files) ===
Total records: 555
Both present (id + mapping_ids): 27
  subset OK   (id in mapping_ids): 27
  subset FAIL (id not in mapping) : 0

id only (id present, mapping_ids empty): 269
  ├─ mapping key missing (None): 254
  └─ mapping key present but no valid ids: 15
mapping only (mapping_ids present, id empty): 0
both empty/missing: 259

ID-only samples:
{'file_id': '1c148301-6d28-46b4-b833-1bda96f2bb8c', 'file_content_id': '28073', 'raw_mapping_type': 'NoneType'}
{'file_id': '1ce4e178-c9f6-4a6b-bd0d-a378cc8a21c0', 'file_content_id': '4e6b3059-5e7e-4f94-b18e-1be48020cab7', 'raw_mapping_type': 'NoneType'}
{'file_id': '1f7af396-f0b1-4bbe-bddd-70aa3deb641f', 'file_content_id': '15376', 'raw_mapping_type': 'NoneType'}
{'file_id': '20b7c54a-cee8-4cf2-a45d-57fee01673b5', 'file_content_id': '236166', 'raw_mapping_type': 'NoneType'}
{'file_id': '24ba936e-e061-4015-9952-81cec3dd0fdc', 'file_content_id': 'fbd0a029-2e7a-45a4-b430-1fb5034c42b0', 'raw_mapping_t

### 1.6 Cross-entity references → `:People` and `:Site`

In [12]:
# Collect People and Site IDs from raw dump for cross-referencing
people_ids = set()
site_ids   = set()
with open(RAW_FILE) as f:
    for line in f:
        obj = json.loads(line)
        ot = obj.get("object_type")
        if ot == "people":
            people_ids.add(obj["id"])
        elif ot == "site":
            site_ids.add(obj["id"])

print(f"People in dump : {len(people_ids)}")
print(f"Sites in dump  : {len(site_ids)}")
print()

# createdbyid → People (file_owner_id == createdbyid for 100% of records — one check suffices)
creator_ids      = set(s["createdbyid"] for s in file_items if s.get("createdbyid"))
missing_creators = creator_ids - people_ids
print(f"Unique createdbyid            : {len(creator_ids)}")
print(f"Missing from people           : {len(missing_creators)}"
      + (" ✅" if not missing_creators else f"  ⚠️ {missing_creators}"))
print()

# file_site_mapping → Site
all_mapping_ids = set(
    sid
    for s in file_items
    for sid in (s.get("file_site_mapping") or [])
)
missing_sites = all_mapping_ids - site_ids
print(f"Unique site IDs in file_site_mapping : {len(all_mapping_ids)}")
print(f"Missing from sites                   : {len(missing_sites)}"
      + (" ✅" if not missing_sites else f"  ⚠️ {missing_sites}"))

People in dump : 954
Sites in dump  : 45

Unique createdbyid            : 57
Missing from people           : 0 ✅

Unique site IDs in file_site_mapping : 28
Missing from sites                   : 1  ⚠️ {None}


### 1.7 File size & page count stats

In [13]:
# file_size
sizes = [s.get("file_size") for s in file_items
         if isinstance(s.get("file_size"), (int, float))]
print(f"file_size — present: {len(sizes)}/{len(file_items)}")
if sizes:
    print(f"  min: {min(sizes):,}  max: {max(sizes):,}  mean: {int(sum(sizes)/len(sizes)):,} bytes")

print()

# file_page_count
page_counts = [s.get("file_page_count") for s in file_items
               if isinstance(s.get("file_page_count"), int)]
print(f"file_page_count — present: {len(page_counts)}/{len(file_items)}")
if page_counts:
    print(f"  min: {min(page_counts)}  max: {max(page_counts)}  mean: {sum(page_counts)/len(page_counts):.1f}")

file_size — present: 550/555
  min: 4  max: 698,844,120  mean: 9,111,382 bytes

file_page_count — present: 300/555
  min: 1  max: 256  mean: 19.1


### 1.8 Sparse / internal fields overview
Quick summary of fields that are almost always null/empty or are internal processing artefacts.

In [14]:
n = len(file_items)

sparse_fields = [
    "file_title",
    "file_author",
    "file_creation_date",
    "file_language",
    "file_keywords",
    "file_transcript",
    "file_content_id",
    "file_external_id",
    "file_thumbnail_url",
    "extracted_info",
    "_aggregated_locations",
    "_s3_orchestrator_processing_file_key",
    "_s3_orchestrator_processing_bucket_key",
]

print(f"{'Field':<45} {'Has value':>10} / {n}")
print("-" * 60)
for k in sparse_fields:
    has_val = sum(
        1 for s in file_items
        if s.get(k) is not None
        and s.get(k) != []
        and s.get(k) != ""
    )
    print(f"{k:<45} {has_val:>10}")

print("\nfile_content_mapping is analysed separately in section 1.5.")

Field                                          Has value / 555
------------------------------------------------------------
file_title                                             0
file_author                                            0
file_creation_date                                     0
file_language                                          0
file_keywords                                          0
file_transcript                                        0
file_content_id                                      296
file_external_id                                     198
file_thumbnail_url                                   121
extracted_info                                         4
_aggregated_locations                                303
_s3_orchestrator_processing_file_key                 340
_s3_orchestrator_processing_bucket_key               340

file_content_mapping is analysed separately in section 1.5.


### 1.9 Full DataFrame overview of key fields

In [15]:
key_cols = [
    "id",
    "file_name",
    "file_type",
    "file_mime_type",
    "file_provider",
    "is_active",
    "is_deleted",
    "file_size",
    "file_page_count",
    "file_content_id",
    "file_content_mapping",
    "file_site_id",
    "file_site_name",
    "createdbyid",
    "createddate",
    "lastmodifieddate",
    "file_url",
]

file_df = pd.DataFrame([{k: s.get(k) for k in key_cols} for s in file_items])

# Preview normalized target key
file_df["file_content_mappings"] = [
    build_content_mappings(r.get("file_content_id"), r.get("file_content_mapping"))
    for r in file_items
]

print(f"Shape: {file_df.shape}")
file_df[[
    "id", "file_name", "file_content_id", "file_content_mapping", "file_content_mappings"
]].head(5)

Shape: (555, 18)


,id,file_name,file_content_id,file_content_mapping,file_content_mappings
0,1c148301-6d28-46b4-b833-1bda96f2bb8c,Ragan -LumApps-Webinar-8.22.24.pdf,28073,None,[28073]
1,1ce4e178-c9f6-4a6b-bd0d-a378cc8a21c0,Simpplr iOS App ACR International Edition - Se...,4e6b3059-5e7e-4f94-b18e-1be48020cab7,None,[4e6b3059-5e7e-4f94-b18e-1be48020cab7]
2,1e0cb7d5-aaf6-4a20-8620-b5c9ab3885fb,Acko Onboarding deck (1).pdf,NaN,None,[]
3,1e120631-b16d-47be-90db-5101ae3be46d,Acko Health_Daycare Procedures (1).pdf,NaN,None,[]
4,1e4fa25b-f44e-40c8-a8aa-30bfc3d8749e,SIM_Pol14_ConfigMgmt&SADM_v5.0_2024.pdf,NaN,None,[]


---
## 🔍 Exploration Summary

### Data overview (non-image files only)
- **555 non-image file records** (7,322 images excluded — no semantic value in the graph)
- `file_provider`: `intranet` and `googledrive`
- `file_site_mapping` cardinality: 6 orphans · ~524 single-site · 25 multi-site
  → one `BELONGS_TO_SITE` edge per entry in the list
- All `createdbyid` values resolve to existing `:People` nodes ✅
- All `file_site_mapping` site IDs resolve to existing `:Site` nodes ✅
- `createdbyid == file_owner_id` for 100% of records → only `CREATED` relationship needed

### Unified content key normalization
- Normalized output uses **one key**: `file_content_mappings`
- `file_content_mappings = union(file_content_mapping, file_content_id)` after cleaning
- Cleaning rules: remove nulls, non-lists/non-strings, empty strings, `@@...`, deduplicate
- `file_content_mappings` is preserved in `files.jsonl` and deferred to content pipeline
- No content-edge ingestion happens in file pipeline

### Ignored / excluded fields
- `file_data`, `file_s3_path`, `file_thumbnail_url`, `file_external_id` — not needed
- `_s3_orchestrator_processing_file_key`, `_s3_orchestrator_processing_bucket_key` — internal artefacts
- `autocomplete`, `file_site_name`, `file_site_type`, `file_site_is_active`, `file_owner_name` — redundant
- `file_title`, `file_author`, `file_creation_date`, `file_language` — 100% null
- `file_keywords`, `file_transcript` — always empty lists
- `file_size`, `file_page_count` — excluded from node
- `file_is_image` — excluded from extraction entirely

### Finalized `:File` node fields
`id` · `file_name` · `file_type` · `file_mime_type` · `file_provider` ·
`is_active` · `is_deleted` · `file_url` ·
`_aggregated_locations` · `createddate` · `lastmodifieddate`

### Finalized relationships
| Relationship | Direction | Source field |
|---|---|---|
| `CREATED` | `(People)→(File)` | `createdbyid` |
| `BELONGS_TO_SITE` | `(File)→(Site)` | `file_site_mapping` (one edge per site ID) |

`file_content_mappings` is intentionally kept only in normalized JSON for future content reconciliation.

## 2 · Extract & Normalise

In [16]:
from src.extractors.file_extractor import extract_files, write_normalized

result = extract_files(RAW_FILE)

print(f"Files extracted : {len(result['files'])}")
print(f"Images skipped  : {result['num_skipped_images']}")
print(f"Orphan files    : {result['num_orphan_files']}  (no site mapping — ingested without BELONGS_TO_SITE edge)")
print(f"file_content_mappings non-empty: {result['num_nonempty_content_mappings']}")
print(f"file_content_mappings empty    : {result['num_empty_content_mappings']}")

write_normalized(result, NORMALIZED_DIR)

Files extracted : 555
Images skipped  : 7322
Orphan files    : 18  (no site mapping — ingested without BELONGS_TO_SITE edge)
file_content_mappings non-empty: 296
file_content_mappings empty    : 259
Written 555 file records → /home/ubuntu/graph_experiments/data/normalized/files.jsonl


## 3 · Analyse Extraction Results

In [17]:
# Load normalised files.jsonl
files_norm = []
with open(NORMALIZED_DIR / "files.jsonl") as f:
    for line in f:
        files_norm.append(json.loads(line))

files_df = pd.DataFrame(files_norm)
print(f"Normalised file records: {len(files_df)}")
print(f"Columns: {list(files_df.columns)}")
files_df.head(3)


Normalised file records: 555
Columns: ['id', 'file_name', 'file_type', 'file_mime_type', 'file_provider', 'is_active', 'is_deleted', 'file_url', '_aggregated_locations', 'createddate', 'lastmodifieddate', 'createdbyid', 'file_site_mapping', 'file_content_mappings']


,id,file_name,file_type,file_mime_type,file_provider,is_active,is_deleted,file_url,_aggregated_locations,createddate,lastmodifieddate,createdbyid,file_site_mapping,file_content_mappings
0,1c148301-6d28-46b4-b833-1bda96f2bb8c,Ragan -LumApps-Webinar-8.22.24.pdf,PDF,application/pdf,intranet,True,False,/content/r/1c148301-6d28-46b4-b833-1bda96f2bb8c,[global],2024-08-26T19:40:36.312Z,2024-08-26T19:40:36.312Z,3a331b59-add7-4198-a9a8-78c11af00261,[39b3d853-fcc5-43cd-a508-d77f56bb0b02],[28073]
1,1ce4e178-c9f6-4a6b-bd0d-a378cc8a21c0,Simpplr iOS App ACR International Edition - Se...,PDF,application/pdf,intranet,True,False,/content/r/1ce4e178-c9f6-4a6b-bd0d-a378cc8a21c0,[global],2025-09-24T04:16:32.184Z,2025-09-24T04:16:32.184Z,6a64c628-7c93-4017-880d-52b8de9b6f3e,[05f54eb2-3eab-435c-9ebd-67c3111f9499],[4e6b3059-5e7e-4f94-b18e-1be48020cab7]
2,1e0cb7d5-aaf6-4a20-8620-b5c9ab3885fb,Acko Onboarding deck (1).pdf,PDF,application/pdf,intranet,True,False,/content/r/1e0cb7d5-aaf6-4a20-8620-b5c9ab3885fb,[india],2024-04-04T19:00:00.823Z,2024-04-04T19:00:18.626Z,3f1dcf21-0489-46de-a743-1a47b506e795,"[00578996-5d88-4a63-bb5b-aabbbb51a0b6, a45f90c...",[]


In [18]:
print("=== file_type distribution ===")
for k, v in Counter(r.get("file_type") for r in files_norm).most_common():
    print(f"  {repr(k)}: {v}")

print()
print("=== file_provider distribution ===")
print(Counter(r.get("file_provider") for r in files_norm))

print()
print("=== is_active distribution ===")
print(Counter(r.get("is_active") for r in files_norm))

print()
print("=== is_deleted distribution ===")
print(Counter(r.get("is_deleted") for r in files_norm))

=== file_type distribution ===
  'PDF': 273
  None: 204
  'PPTX': 29
  'DOCX': 11
  'XLSX': 8
  'TXT': 5
  'PPT': 4
  'ZIP': 4
  'HTML': 3
  'XLS': 3
  'PHP': 2
  'WAV': 2
  'JS': 2
  'CSV': 1
  'POTX': 1
  'JSON': 1
  'MPGA': 1
  'GSLIDES': 1

=== file_provider distribution ===
Counter({'intranet': 357, 'googledrive': 198})

=== is_active distribution ===
Counter({True: 530, False: 25})

=== is_deleted distribution ===
Counter({False: 473, True: 82})


In [19]:
# Cross-reference: verify FKs against People + Site + unified content mappings quality
print("=== Cross-entity reference check ===")
print()

# createdbyid → People
creator_ids_norm = set(r["createdbyid"] for r in files_norm if r.get("createdbyid"))
missing_creators_norm = creator_ids_norm - people_ids
print(f"Unique createdbyid  : {len(creator_ids_norm)}")
print(f"Missing from people : {len(missing_creators_norm)}"
      + (" ✅" if not missing_creators_norm else f"  ⚠️ {missing_creators_norm}"))

print()

# file_site_mapping → Site (all IDs across all lists)
all_site_refs = set()
for r in files_norm:
    for sid in (r.get("file_site_mapping") or []):
        all_site_refs.add(sid)

missing_site_refs = all_site_refs - site_ids
print(f"Unique site IDs in file_site_mapping : {len(all_site_refs)}")
print(f"Missing from sites                   : {len(missing_site_refs)}"
      + (" ✅" if not missing_site_refs else f"  ⚠️ {missing_site_refs}"))

print()

# Orphan files
orphan_count = sum(1 for r in files_norm if not r.get("file_site_mapping"))
print(f"Orphan files (empty file_site_mapping): {orphan_count}")

print()

# file_content_mappings quality (normalized)
mappings_nonempty = sum(1 for r in files_norm if r.get("file_content_mappings"))
mappings_empty = len(files_norm) - mappings_nonempty

invalid_normalized_items = 0
for r in files_norm:
    vals = r.get("file_content_mappings") or []
    for value in vals:
        if not isinstance(value, str) or not value.strip() or value.startswith("@@"):
            invalid_normalized_items += 1

print(f"Files with non-empty file_content_mappings : {mappings_nonempty}/{len(files_norm)}")
print(f"Files with empty file_content_mappings     : {mappings_empty}/{len(files_norm)}")
print(f"Invalid values inside file_content_mappings: {invalid_normalized_items}")
print("(file_content_mappings is kept in files.jsonl and deferred to content pipeline)")

=== Cross-entity reference check ===

Unique createdbyid  : 57
Missing from people : 0 ✅

Unique site IDs in file_site_mapping : 27
Missing from sites                   : 0 ✅

Orphan files (empty file_site_mapping): 18

Files with non-empty file_content_mappings : 296/555
Files with empty file_content_mappings     : 259/555
Invalid values inside file_content_mappings: 0
(file_content_mappings is kept in files.jsonl and deferred to content pipeline)


### 3 · Extraction summary

| Metric | Value |
|---|---|
| Images excluded (`file_is_image=True`) | 7,322 |
| **Non-image files extracted** | **555** |
| Orphan files (no `file_site_mapping`) | see output above |
| Unique `createdbyid` references | all resolve to `:People` ✅ |
| Unique site IDs in `file_site_mapping` | all resolve to `:Site` ✅ |
| Files with non-empty `file_content_mappings` | see output above |
| Files with empty `file_content_mappings` | see output above |

**`:File` node fields:** `id` · `file_name` · `file_type` · `file_mime_type` · `file_provider` · `is_active` · `is_deleted` · `file_url` · `_aggregated_locations` · `createddate` · `lastmodifieddate`

**Stored in normalized JSONL only (not ingested in Neo4j in this pipeline):**
- `file_content_mappings` (clean list of content IDs from `file_content_mapping ∪ file_content_id`)

**Relationships:**
- `(People)-[:CREATED]->(File)` — one per file with a `createdbyid`
- `(File)-[:BELONGS_TO_SITE]->(Site)` — one per entry in `file_site_mapping`

## 4 · Ingest into Neo4j

In [20]:
from src.loaders.file_loader import load_file_graph
from src.utils.neo4j_client import Neo4jClient

client = Neo4jClient(uri=NEO4J_URI, user=NEO4J_USER, password=NEO4J_PASSWORD)
load_file_graph(client, NORMALIZED_DIR, batch=True)


✅ Constraints ensured.
✅ Files loaded: 555
ℹ️  file_content_mappings is preserved in files.jsonl and deferred to content ingestion.
✅ CREATED relationships: 555
✅ All file_site_mapping site IDs resolved.
✅ BELONGS_TO_SITE relationships: 561

✅ File graph fully loaded.


## 5 · Post-Ingestion Verification

In [21]:
# Node counts
node_counts = client.query("""
    MATCH (n)
    WHERE n:File OR n:People OR n:Site
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY label
""")
print("=== Node counts ===")
for row in node_counts:
    print(f"  :{row['label']:<15} {row['count']:>6}")

print()

# Relationship counts
rel_counts = client.query("""
    MATCH ()-[r]->()
    WHERE type(r) IN ['CREATED', 'BELONGS_TO_SITE']
    RETURN type(r) AS rel_type, count(r) AS count
    ORDER BY rel_type
""")
print("=== Relationship counts ===")
for row in rel_counts:
    print(f"  :{row['rel_type']:<25} {row['count']:>6}")


=== Node counts ===
  :File               555
  :People             954
  :Site                45

=== Relationship counts ===
  :BELONGS_TO_SITE              561
  :CREATED                      600


In [22]:
# file_type distribution in Neo4j
print("=== :File file_type distribution ===")
rows = client.query("""
    MATCH (f:File)
    RETURN coalesce(f.file_type, '(null)') AS file_type, count(f) AS cnt
    ORDER BY cnt DESC
""")
for row in rows:
    print(f"  {row['file_type']:<20} {row['cnt']:>5}")

print()

# file_provider distribution
print("=== :File file_provider distribution ===")
rows = client.query("""
    MATCH (f:File)
    RETURN coalesce(f.file_provider, '(null)') AS provider, count(f) AS cnt
    ORDER BY cnt DESC
""")
for row in rows:
    print(f"  {row['provider']:<20} {row['cnt']:>5}")


=== :File file_type distribution ===
  PDF                    273
  (null)                 204
  PPTX                    29
  DOCX                    11
  XLSX                     8
  TXT                      5
  PPT                      4
  ZIP                      4
  HTML                     3
  XLS                      3
  PHP                      2
  WAV                      2
  JS                       2
  CSV                      1
  POTX                     1
  JSON                     1
  MPGA                     1
  GSLIDES                  1

=== :File file_provider distribution ===
  intranet               357
  googledrive            198


In [23]:
# Orphan check — :File nodes with no BELONGS_TO_SITE edge
orphan_check = client.query("""
    MATCH (f:File)
    WHERE NOT (f)-[:BELONGS_TO_SITE]->()
    RETURN count(f) AS orphan_files
""")
print(f"Orphan :File nodes (no BELONGS_TO_SITE edge): {orphan_check[0]['orphan_files']}")

print()

# Missing CREATED edge — :File nodes with no CREATED edge incoming
no_creator = client.query("""
    MATCH (f:File)
    WHERE NOT ()-[:CREATED]->(f)
    RETURN count(f) AS no_creator
""")
print(f":File nodes with no CREATED edge: {no_creator[0]['no_creator']}")

print()

# Top 10 sites by file count
print("=== Top 10 Sites by :File count ===")
top_sites = client.query("""
    MATCH (f:File)-[:BELONGS_TO_SITE]->(s:Site)
    RETURN s.site_name AS site, count(f) AS file_count
    ORDER BY file_count DESC
    LIMIT 10
""")
for row in top_sites:
    print(f"  {row['site']:<40} {row['file_count']:>5}")

print()

# Top 10 people by files created
print("=== Top 10 People by files CREATED ===")
top_creators = client.query("""
    MATCH (p:People)-[:CREATED]->(f:File)
    RETURN p.user_name AS person, count(f) AS file_count
    ORDER BY file_count DESC
    LIMIT 10
""")
for row in top_creators:
    print(f"  {row['person']:<40} {row['file_count']:>5}")


Orphan :File nodes (no BELONGS_TO_SITE edge): 18

:File nodes with no CREATED edge: 0

=== Top 10 Sites by :File count ===
  Strategy & Value Consulting                130
  IT & Security Team                          57
  Marketing Team                              55
  Product Team                                54
  Professional Services                       34
  Life at Simpplr                             27
  Revenue Team                                24
  Engineering Team                            24
  Simpplr One Upgrade                         21
  India                                       20

=== Top 10 People by files CREATED ===
  Christy Schoon                             130
  Melissa Hewitt                              61
  Huy Nguyen                                  56
  Christine Robertson                         39
  Rachel Smith                                34
  Divyankar Bhargav                           22
  Viviana Serrato Sotelo                      17
  Ka

In [24]:
client.close()
print("Neo4j connection closed.")


Neo4j connection closed.
